In [1]:
import pandas as pd
import numpy as np

#carregando o dataset
file_inmet = '/content/drive/MyDrive/Pivic/dados/Dados/Institudo de Metereologia/dados 2022/INMET_NE_PB_A313_CAMPINA GRANDE_01-01-2022_A_31-12-2022.CSV'

df_clima = pd.read_csv(file_inmet,
                       sep=';',
                       decimal=',',
                       skiprows=8,
                       encoding='latin1')
# selecionando as colunas que serão úteis
colunas_map = {
    'Data': 'DATA',
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'CHUVA_TOTAL',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)': 'TEMP_MEDIA',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'TEMP_MAX',
    'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'TEMP_MIN',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'UMIDADE'
}
df_clima = df_clima.rename(columns=colunas_map)[colunas_map.values()]


#tratando irregularidade

df_clima['DATA'] = pd.to_datetime(df_clima['DATA'], format='%Y/%m/%d')


#transformando o valor em datetime para semana, pq coloquei os valores na tabela do dataset da epidemiologia em semanas
df_clima['SEMANA'] = df_clima['DATA'].dt.isocalendar().week


#salvando os parâmetros por semana
df_clima_semanal = df_clima.groupby('SEMANA').agg({
    'CHUVA_TOTAL': 'sum',
    'TEMP_MEDIA': 'mean',
    'TEMP_MAX': 'max',
    'TEMP_MIN': 'min',
    'UMIDADE': 'mean'
}).reset_index()

df_clima_semanal = df_clima_semanal.round(2)



df_dengue = pd.read_csv('/content/drive/MyDrive/Pivic/tratamento dados/dados tratados/epidemiologia/dengue2022_tratado.csv')

#transformando os valores em inteiro para que o cruzamento das bases não dê bronca
df_dengue['SEMANA EPDM'] = pd.to_numeric(df_dengue['SEMANA EPDM'], errors='coerce')

#criando uma coluna para agregar os casos em quantidade por semana
df_casos_semanal = df_dengue.groupby('SEMANA EPDM').size().reset_index(name='TOTAL_CASOS')
df_casos_semanal = df_casos_semanal.rename(columns={'SEMANA EPDM': 'SEMANA'})



#fazendo o merge propriamente dito, utilizando o inner, serve pra utilizar apenas colunas que estão presente nas duas tabelas
df_final = pd.merge(df_clima_semanal, df_casos_semanal, on='SEMANA', how='inner')

#salvandotratamento_clima2025.
#df_final.to_csv('dataset_clima/semana_epdm.csv', index=False)

df_final.head(10)

,SEMANA,CHUVA_TOTAL,TEMP_MEDIA,TEMP_MAX,TEMP_MIN,UMIDADE,TOTAL_CASOS
0,4,40.2,24.60,32.3,18.6,79.90,1
1,5,1.0,25.23,31.8,21.2,77.02,2
2,6,0.0,25.67,32.7,21.6,75.08,3
3,7,2.8,25.31,33.7,21.6,77.00,3
4,9,15.4,25.16,33.3,21.2,79.08,1
5,10,1.4,25.43,33.0,20.7,77.17,4
6,11,46.4,25.05,32.0,21.1,82.89,7
7,12,133.0,23.72,30.3,19.7,90.95,11
8,13,11.4,24.66,31.1,21.7,83.89,2
9,14,13.2,24.45,31.1,20.4,84.54,3


In [3]:
df_final.to_csv('dataset_clima_semana_epdm2022.csv', index=False)
df_clima_semanal.to_csv('dataset_clima_2022_tratato.csv',index=False)